# 31 Combining studies, planning a study, surveys and designs

Real-data checks of the groups "Combining studies", "Planning a study" and "Planning and surveys". This notebook holds
only simplydata: data read with `sdata.table`, the words, the numbers read back with simplydata's own words (through
`Result`), and the comparison with `references/31_combining_planning_surveys.csv`; each comparison prints its
"Source:" with a link.

Printed references: Viechtbauer (2010) for the BCG random-effects model, the pwr vignette for two sample sizes, Lumley
(2010) for the stratified api mean. Elsewhere `scripts/references/31_combining_planning_surveys.py` runs the help-page
example of the library simplydata calls once (metafor, power.t.test, gsDesign, powerSurvEpi, pwr, pwrss, TOSTER, SciPy
stats.power with the same seed, survey, agricolae, blockrand). Tolerances: printed values to half a unit of their last
digit; relative 1e-10 for closed formulas, 1e-6 for iterative fits, 1e-12 for the same seed; lists compared exactly.

The numbers the forest and funnel plots draw are read back with `chart.get_numbers_drawn()` (section 6).

In [1]:
import simplydata as sdata

from reference_checks import Comparisons, Result, quiet

cmp = Comparisons("31_combining_planning_surveys")
DATA = "../data/"
FETCHED = "../data/fetched/"  # the datasets this repository does not host (written by scripts/get_data.py)


def combined(result, ratio):
    """The combined row of a combine_studies result; ratios are compared on the log scale (with simplydata's take_log)."""
    t = Result(result, where={"row kind": "combined"})
    row = t.row(1)
    ends = [row["estimate"], row["estimate 95 % CI low"], row["estimate 95 % CI high"]]
    if ratio == "log":
        ends = list(sdata.numbers(ends).take_log())
    return {"estimate": ends[0], "low": ends[1], "high": ends[2], "standard error": row["standard error"], "p-value": row["p-value"],
            "tau squared": row["tau squared"], "I squared %": row["I squared %"], "Q": row["Q"]}

## 1. Combining studies

BCG trials: Viechtbauer (2010) prints the REML random-effects model of the log risk ratios (-0.7145, SE 0.1798, CI
-1.0669 to -0.3622, tau^2 0.3132, I^2 92.22 %, Q 152.2330); `interval_by="Hartung-Knapp"` (with the ad hoc rule
simplydata documents, metafor `test = "adhoc"`); Peto's method; estimates given with their interval columns or with
standard errors; ratios of averages; correlations; rates.

In [2]:
bcg = sdata.table(FETCHED + "dat_bcg.csv", separator=",")
quiet(bcg.add_column, "vaccinated", by_adding=["tpos", "tneg"])
quiet(bcg.add_column, "unvaccinated", by_adding=["cpos", "cneg"])
COUNTS = dict(names="author", events=["tpos", "cpos"], out_of=["vaccinated", "unvaccinated"])
for interval_by in ("normal", "Hartung-Knapp"):
    ours = combined(quiet(bcg.combine_studies, show="risk ratios", test_method="reml", interval_by=interval_by, **COUNTS), "log")
    printed = ("estimate", "standard error", "low", "high", "tau squared", "I squared %", "Q")   # what Viechtbauer (2010) prints
    run = ("estimate", "standard error", "low", "high", "p-value", "tau squared")                 # what the metafor run gives back
    ours = {k: ours[k] for k in (printed if interval_by == "normal" else run)}
    cmp.check_all("BCG trials (metadat)", f'combine_studies (show="risk ratios", test_method="reml", interval_by="{interval_by}")', ours)
peto = combined(quiet(bcg.combine_studies, show="odds ratios", test_method="peto", **COUNTS), "log")
cmp.check_all("BCG trials (metadat)", 'combine_studies (show="odds ratios", test_method="peto")',
              {k: peto[k] for k in ("estimate", "standard error", "low", "high")})

hackshaw = sdata.table(FETCHED + "dat_hackshaw1998.csv", separator=",")
for old, new in (("or", "odds ratio"), ("or.lb", "odds ratio 95 % CI low"), ("or.ub", "odds ratio 95 % CI high")):
    quiet(hackshaw.rename_column, old, new)
quiet(hackshaw.add_column, "se", "vi ** 0.5")
CASES = [("Hackshaw (1997), passive smoking", hackshaw, dict(names="author", estimates="odds ratio"), "log",
          "combine_studies (estimates= with interval columns, a ratio)"),
         ("Hackshaw (1997), passive smoking", hackshaw, dict(names="author", estimates="yi", standard_errors="se", estimates_are_ratios=False), None,
          "combine_studies (standard_errors=, estimates_are_ratios=False)"),
         ("Curtis et al. (1998), CO2", sdata.table(FETCHED + "dat_curtis1998.csv", separator=","),
          dict(averages=["m1i", "m2i"], standard_deviations=["sd1i", "sd2i"], counts=["n1i", "n2i"], show="ratios of averages"), "log",
          'combine_studies (averages=, standard_deviations=, show="ratios of averages")'),
         ("Hart et al. (1999), warfarin", sdata.table(FETCHED + "dat_hart1999.csv", separator=","), dict(names="study", events=["x1i", "x2i"], time=["t1i", "t2i"]), "log",
          "combine_studies (events=, time=)")]
for dataset, data, options, scale, check in CASES:
    ours = combined(quiet(data.combine_studies, **options), scale)
    cmp.check_all(dataset, check, {k: ours[k] for k in ("estimate", "standard error", "low", "high", "tau squared")})
# correlations are combined on Fisher's z: the estimate and interval are turned back to correlations, the standard
# error and tau^2 stay on the z scale (so does the reference)
mcd = combined(quiet(sdata.table(FETCHED + "dat_mcdaniel1994.csv", separator=",").combine_studies, correlations="ri", counts="ni"), None)
cmp.check_all("McDaniel et al. (1994), interviews", "combine_studies (correlations=, counts=)",
              {k: mcd[k] for k in ("estimate", "standard error", "low", "high", "tau squared")})

agrees  BCG trials (metadat) | combine_studies (show="risk ratios", test_method="reml", interval_by="normal") | estimate: simplydata -0.714532342158, Viechtbauer (2010), Conducting meta-analyses in R with the metafor package, J. Stat. Softw. 36(3), section 4 -0.7145; difference 3.23e-05 (allowed 5e-05)
      Source: Viechtbauer (2010), Conducting meta-analyses in R with the metafor package, J. Stat. Softw. 36(3), section 4 (printed) - https://doi.org/10.18637/jss.v036.i03
agrees  BCG trials (metadat) | combine_studies (show="risk ratios", test_method="reml", interval_by="normal") | standard error: simplydata 0.179781516105, Viechtbauer (2010), Conducting meta-analyses in R with the metafor package, J. Stat. Softw. 36(3), section 4 0.1798; difference 1.85e-05 (allowed 5e-05)
      Source: Viechtbauer (2010), Conducting meta-analyses in R with the metafor package, J. Stat. Softw. 36(3), section 4 (printed) - https://doi.org/10.18637/jss.v036.i03
agrees  BCG trials (metadat) | combine_stu

explained difference – precision limit (C29)  Hackshaw (1997), passive smoking | combine_studies (standard_errors=, estimates_are_ratios=False) | estimate: simplydata 0.218889231613, R metafor escalc + rma (REML, test = adhoc) (metafor 5.2.1) 0.218889774308; relative difference 2.48e-06 (allowed 1e-06)
      why: References run with R's default stopping rule; simplydata now converges tighter (its A7 fix) - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c29-references-run-with-rs-default-stopping-rule-simplydata-now-converges-tighter-its-a7-fix
      Source: R metafor escalc + rma (REML, test = adhoc) (metafor 5.2.1) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/rma.uni.html
agrees  Hackshaw (1997), passive smoking | combine_studies (standard_errors=, estimates_are_ratios=False) | estimate: simplydata 0.218889231613, R metafor escalc + rma (REML, test = adhoc, tol 1e-12) (metafor 5.2.1) 0.218889231609; relative difference 1.94e-11 (all

explained difference – precision limit (C29)  Curtis et al. (1998), CO2 | combine_studies (averages=, standard_deviations=, show="ratios of averages") | estimate: simplydata 0.25396167517, R metafor escalc + rma (REML, test = adhoc) (metafor 5.2.1) 0.253962265876; relative difference 2.33e-06 (allowed 1e-06)
      why: References run with R's default stopping rule; simplydata now converges tighter (its A7 fix) - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c29-references-run-with-rs-default-stopping-rule-simplydata-now-converges-tighter-its-a7-fix
      Source: R metafor escalc + rma (REML, test = adhoc) (metafor 5.2.1) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html
agrees  Curtis et al. (1998), CO2 | combine_studies (averages=, standard_deviations=, show="ratios of averages") | estimate: simplydata 0.25396167517, R metafor escalc + rma (REML, test = adhoc, tol 1e-12) (metafor 5.2.1) 0.253961675165; relative difference 1.

agrees  McDaniel et al. (1994), interviews | combine_studies (correlations=, counts=) | estimate: simplydata 0.233032253056, R metafor escalc + rma (REML, test = adhoc) (metafor 5.2.1) 0.233032270431; relative difference 7.46e-08 (allowed 1e-06)
      Source: R metafor escalc + rma (REML, test = adhoc) (metafor 5.2.1) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html
agrees  McDaniel et al. (1994), interviews | combine_studies (correlations=, counts=) | estimate: simplydata 0.233032253056, R metafor escalc + rma (REML, test = adhoc, tol 1e-12) (metafor 5.2.1) 0.233032253056; relative difference 1.54e-13 (allowed 1e-06)
      Source: R metafor escalc + rma (REML, test = adhoc, tol 1e-12) (metafor 5.2.1) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html
agrees  McDaniel et al. (1994), interviews | combine_studies (correlations=, counts=) | standard error: simplydata 0.0177804505246, R metafor escalc + rma (REML, test =

## 3. Planning a study

Printed in the pwr vignette: a difference of 0.75 with a standard deviation of 2.25 needs n = 142.2466 per group; a
one-sample, one-sided test of d = 0.5/2.25 with 90 % power needs n = 174.7796. Run once: R power.t.test (significance
0.01; clusters), gsDesign (Schoenfeld), powerSurvEpi (Freedman), pwr.f2.test, pwrss (logistic), TOSTER (equivalence)
and SciPy stats.power with the same seed (rank test).

In [3]:
def one(result):
    return Result(result).row(1)


P = "planning settings"
cmp.check(P, "find_how_many_needed (difference=0.75, spread=2.25)", "values per group (exact)",
          one(quiet(sdata.find_how_many_needed, difference=0.75, spread=2.25))["values per group (exact)"])
cmp.check(P, 'find_how_many_needed (for_test="compare_average_to", sides=1, power=0.9)', "values (exact)",
          one(quiet(sdata.find_how_many_needed, for_test="compare_average_to", effect_size=0.5 / 2.25, power=0.9, sides=1))["values per group (exact)"])
cmp.check(P, "find_how_many_needed (effect_size=0.5, significance=0.01)", "values per group (exact)",
          one(quiet(sdata.find_how_many_needed, effect_size=0.5, significance=0.01))["values per group (exact)"])
cmp.check(P, "find_chance_of_finding (effect_size=0.5, values_per_group=30, significance=0.01)", "power",
          one(quiet(sdata.find_chance_of_finding, effect_size=0.5, values_per_group=30, significance=0.01))["power"])
cmp.check(P, 'find_how_many_needed (for_test="compare_survival", effect_size=0.7, percents=[30, 40])', "events (exact)",
          one(quiet(sdata.find_how_many_needed, for_test="compare_survival", effect_size=0.7, percents=[30, 40]))["events (exact)"])
cmp.check(P, 'find_how_many_needed (for_test="compare_survival", test_method="freedman")', "people per group",
          one(quiet(sdata.find_how_many_needed, for_test="compare_survival", effect_size=0.7, percents=[30, 40], test_method="freedman"))["people per group"])
cmp.check(P, 'find_how_many_needed (for_test="explain", effect_size=0.15)', "rows (exact)",
          one(quiet(sdata.find_how_many_needed, for_test="explain", effect_size=0.15, using=["a", "b", "c"]))["rows (exact)"])
cmp.check(P, 'find_how_many_needed (for_test="compare_models", effect_size=0.15, against=)', "rows (exact)",
          one(quiet(sdata.find_how_many_needed, for_test="compare_models", effect_size=0.15, using=["a"], against=["a", "b", "c"]))["rows (exact)"])
cmp.check(P, 'find_chance_of_finding (for_test="explain", effect_size=0.15, using=3, values_per_group=77)', "power",
          one(quiet(sdata.find_chance_of_finding, for_test="explain", effect_size=0.15, using=3, values_per_group=77))["power"])
cmp.check(P, 'find_how_many_needed (for_test="explain", outcome="yes or no", percents=[20], effect_size=1.5)', "values (rounded up)",
          one(quiet(sdata.find_how_many_needed, for_test="explain", outcome="yes or no", using=["dose"], percents=[20], effect_size=1.5))["values"])
cmp.check(P, 'find_chance_of_finding (for_test="explain", outcome="yes or no", percents=[20], effect_size=1.5, values_per_group=200)', "power",
          one(quiet(sdata.find_chance_of_finding, for_test="explain", outcome="yes or no", using=["dose"], percents=[20], effect_size=1.5,
                    values_per_group=200))["power"])
cmp.check(P, "find_chance_of_finding (values_per_group=100, in_clusters_of=20, alike_within_clusters=0.05)", "power",
          one(quiet(sdata.find_chance_of_finding, effect_size=0.5, values_per_group=100, in_clusters_of=20, alike_within_clusters=0.05))["power"])
ranks = Result(quiet(sdata.find_chance_of_finding, effect_size=0.5, values_per_group=[20, 30], based_on="ranks", tries=2000, with_seed=1))
for i, n in enumerate((20, 30), start=1):
    cmp.check(P, 'find_chance_of_finding (based_on="ranks", values_per_group=[20, 30], tries=2000, with_seed=1)', f"power with {n} per group",
              ranks.cell(i, "power"))
equivalence = one(quiet(sdata.find_how_many_needed, same_within=5, spread=10))
cmp.check(P, "find_how_many_needed (same_within=5, spread=10)", "values per group (exact)", equivalence[[k for k in equivalence if "exact" in k][0]])

agrees  planning settings | find_how_many_needed (difference=0.75, spread=2.25) | values per group (exact): simplydata 142.246595845, pwr vignette (Champely et al.), printed output: power.t.test(delta = 0.75, sd = 2.25, power = 0.8) 142.2466; difference 4.16e-06 (allowed 5e-05)
      Source: pwr vignette (Champely et al.), printed output: power.t.test(delta = 0.75, sd = 2.25, power = 0.8) (printed) - https://cran.r-project.org/web/packages/pwr/vignettes/pwr-vignette.html
agrees  planning settings | find_how_many_needed (for_test="compare_average_to", sides=1, power=0.9) | values (exact): simplydata 174.779620979, pwr vignette (Champely et al.), printed output: pwr.t.test(d = 0.5/2.25, power = 0.9, alternative = 'greater', type = 'one.sample') 174.7796; difference 2.10e-05 (allowed 5e-05)
      Source: pwr vignette (Champely et al.), printed output: pwr.t.test(d = 0.5/2.25, power = 0.9, alternative = 'greater', type = 'one.sample') (printed) - https://cran.r-project.org/web/packages/pwr

agrees  planning settings | find_how_many_needed (for_test="compare_survival", effect_size=0.7, percents=[30, 40]) | events (exact): simplydata 246.787104547, R gsDesign nEvents (Schoenfeld) (gsDesign 3.11.0) 246.787104547; relative difference 0.00e+00 (allowed 1e-10)
      Source: R gsDesign nEvents (Schoenfeld) (gsDesign 3.11.0) (run reference) - https://search.r-project.org/CRAN/refmans/gsDesign/html/nEvents.html
agrees  planning settings | find_how_many_needed (for_test="compare_survival", test_method="freedman") | people per group: simplydata 361, R powerSurvEpi ssizeCT.default (Freedman) (powerSurvEpi 0.1.5) 361; difference 0.00e+00 (allowed 0e+00)
      Source: R powerSurvEpi ssizeCT.default (Freedman) (powerSurvEpi 0.1.5) (run reference) - https://search.r-project.org/CRAN/refmans/powerSurvEpi/html/ssizeCT.default.html
agrees  planning settings | find_how_many_needed (for_test="explain", effect_size=0.15) | rows (exact): simplydata 76.7058265578, R pwr pwr.f2.test (pwr 1.3.0) 7

agrees  planning settings | find_how_many_needed (same_within=5, spread=10) | values per group (exact): simplydata 69.1978220019, R TOSTER power_t_TOST (TOSTER 0.8.6) 69.1978220019; relative difference 0.00e+00 (allowed 1e-06)
      Source: R TOSTER power_t_TOST (TOSTER 0.8.6) (run reference) - https://search.r-project.org/CRAN/refmans/TOSTER/html/power_t_TOST.html


1

## 4. Survey designs: the api samples (the help page of `survey::svydesign`)

In [4]:
DESIGNS = [("apistrat (survey)", "apistrat.csv", dict(weights="pw"), 'weights="pw"'),
           ("apistrat (survey)", "apistrat.csv", dict(weights="pw", strata="stype"), 'weights="pw", strata="stype"'),
           ("apistrat (survey)", "apistrat.csv", dict(weights="pw", strata="stype", population_sizes="fpc"), 'weights="pw", strata="stype", population_sizes="fpc"'),
           ("apiclus1 (survey)", "apiclus1.csv", dict(weights="pw", clusters="dnum"), 'weights="pw", clusters="dnum"'),
           ("apiclus1 (survey)", "apiclus1.csv", dict(weights="pw", clusters="dnum", population_sizes="fpc"), 'weights="pw", clusters="dnum", population_sizes="fpc"')]
for dataset, file, options, label in DESIGNS:
    t = sdata.table(FETCHED + file, separator=",")
    quiet(t.use_survey_design, **options)
    average = one(quiet(t.find_average_of, "api00"))
    mean = [k for k in average if k.startswith("average")][0]
    se = [k for k in average if "standard error" in k][0]
    cmp.check_all(dataset, f"use_survey_design ({label}), find_average_of", {"average of api00": average[mean], "standard error": average[se]})
t = sdata.table(FETCHED + "apistrat.csv", separator=",")
quiet(t.use_survey_design, weights="pw", strata="stype", population_sizes="fpc")
model = Result(quiet(t.explain, "api00", using=["ell", "meals"]), key="term")
for term in ("(Intercept)", "ell", "meals"):
    cmp.check_all("apistrat (survey)", "use_survey_design (weights=, strata=, population_sizes=), explain",
                  {f"{term} estimate": model[term, "estimate"], f"{term} standard error": model[term, "standard error"], f"{term} p-value": model[term, "p-value"]})

agrees  apistrat (survey) | use_survey_design (weights="pw"), find_average_of | average of api00: simplydata 662.287363159, R survey svymean (help page designs) (survey 4.5) 662.287363159; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survey svymean (help page designs) (survey 4.5) (run reference) - https://search.r-project.org/CRAN/refmans/survey/html/svydesign.html
agrees  apistrat (survey) | use_survey_design (weights="pw"), find_average_of | standard error: simplydata 9.58542887637, R survey svymean (help page designs) (survey 4.5) 9.58542887637; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survey svymean (help page designs) (survey 4.5) (run reference) - https://search.r-project.org/CRAN/refmans/survey/html/svydesign.html
agrees  apistrat (survey) | use_survey_design (weights="pw", strata="stype"), find_average_of | average of api00: simplydata 662.287363159, R survey svymean (help page designs) (survey 4.5) 662.287363159; relative difference 0.00e+0

agrees  apistrat (survey) | use_survey_design (weights=, strata=, population_sizes=), explain | (Intercept) estimate: simplydata 823.857925625, R survey svyglm (survey 4.5) 823.857925625; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survey svyglm (survey 4.5) (run reference) - https://search.r-project.org/CRAN/refmans/survey/html/svyglm.html
agrees  apistrat (survey) | use_survey_design (weights=, strata=, population_sizes=), explain | (Intercept) standard error: simplydata 8.75949494698, R survey svyglm (survey 4.5) 8.75949494698; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survey svyglm (survey 4.5) (run reference) - https://search.r-project.org/CRAN/refmans/survey/html/svyglm.html
agrees  apistrat (survey) | use_survey_design (weights=, strata=, population_sizes=), explain | (Intercept) p-value: simplydata 2.03404739029e-164, R survey svyglm (survey 4.5) 2.03404739029e-164; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survey svyglm (s

## 5. Experimental designs and randomisation lists with a seed (R agricolae `design.split` and blockrand, same seed)

In [5]:
for seed in (4, 11):
    design = Result(quiet(sdata.make_design_table, {"a": [1, 2], "b": ["x", "y", "z"]}, of_kind="split plot", whole_plots="a", repeats=3, with_seed=seed))
    check = f'make_design_table (of_kind="split plot", whole_plots="a", repeats=3, with_seed={seed})'
    for column in ("a", "b", "block"):
        cmp.check_vector("designs (same seed)", check, column, design[column])
for how_many, seed in ((20, 3), (30, 8)):
    listed = Result(quiet(sdata.make_random_assignments, how_many=how_many, groups=["A", "B"], in_blocks_of=[4, 6], with_seed=seed))
    check = f"make_random_assignments (how_many={how_many}, in_blocks_of=[4, 6], with_seed={seed})"
    cmp.check_vector("designs (same seed)", check, "group", listed["group"])
    cmp.check_vector("designs (same seed)", check, "block size", listed["block size"])

agrees  designs (same seed) | make_design_table (of_kind="split plot", whole_plots="a", repeats=3, with_seed=4) | a [1]: simplydata 2, R agricolae design.split (same seed) (agricolae 1.3.8) 2; difference 0.00e+00 (allowed 0e+00)
      Source: R agricolae design.split (same seed) (agricolae 1.3.8) (run reference) - https://search.r-project.org/CRAN/refmans/agricolae/html/design.split.html
agrees  designs (same seed) | make_design_table (of_kind="split plot", whole_plots="a", repeats=3, with_seed=4) | a [2]: simplydata 2, R agricolae design.split (same seed) (agricolae 1.3.8) 2; difference 0.00e+00 (allowed 0e+00)
      Source: R agricolae design.split (same seed) (agricolae 1.3.8) (run reference) - https://search.r-project.org/CRAN/refmans/agricolae/html/design.split.html
agrees  designs (same seed) | make_design_table (of_kind="split plot", whole_plots="a", repeats=3, with_seed=4) | a [3]: simplydata 2, R agricolae design.split (same seed) (agricolae 1.3.8) 2; difference 0.00e+00 (allo

## 6. The numbers the forest and funnel plots draw: `chart.get_numbers_drawn()`

The BCG random-effects result (risk ratios, REML with simplydata's default Hartung-Knapp interval) drawn as a forest
plot and a funnel plot; each chart gives back what it drew as a table (layer, kind, x, y, low, high). Forest plot: each
study's dot and its interval (R metafor `escalc`, normal 95 % interval), the dashed line at 1, the diamond (its left
end, middle and right end: the combined risk ratio and its interval) and the prediction interval line (metafor `rma` +
`predict`). Funnel plot: each study at (risk ratio, standard error), the line at the combined effect and the pseudo
95 % limits, as metafor `funnel()` draws them.

In [6]:
def drawn(chart, **where):
    """The numbers a chart drew (chart.get_numbers_drawn()), the rows of one layer picked by their columns."""
    return Result(quiet(chart.get_numbers_drawn), where=where)


D = "BCG trials (metadat)"
bcg_result = quiet(bcg.combine_studies, show="risk ratios", test_method="reml", **COUNTS)

chart = bcg_result.start_chart()
quiet(chart.forest_plot, x="estimate", y="study", line_at=1, log_scale=True)
FOREST = "chart forest_plot (line_at=1, log_scale=)"
dots, bars, diamond, lines = drawn(chart, kind="dot"), drawn(chart, kind="segment"), drawn(chart, kind="shape"), drawn(chart, kind="line")
cmp.check_vector(D, FOREST, "study dot (risk ratio)", dots["x"])
cmp.check_vector(D, FOREST, "study interval low", bars["low"])
cmp.check_vector(D, FOREST, "study interval high", bars["high"])
# the line at 1 runs over the whole height (no y in the data); the prediction interval is a line at a height
cmp.check_vector(D, FOREST, "line at", [x for x, y in zip(lines["x"], lines["y"]) if y != y])
prediction = [x for x, y in zip(lines["x"], lines["y"]) if y == y]
corners = list(zip(diamond["x"], diamond["y"]))
cmp.check_all(D, FOREST, {"diamond: combined risk ratio": max(corners, key=lambda c: c[1])[0], "diamond: left end (95 % low)": min(diamond["x"]),
                          "diamond: right end (95 % high)": max(diamond["x"]), "prediction interval low": min(prediction),
                          "prediction interval high": max(prediction)})

chart = bcg_result.start_chart()
quiet(chart.draw_funnel_plot, log_scale=True)
FUNNEL = "chart draw_funnel_plot (log_scale=)"
dots, lines = drawn(chart, kind="dot"), drawn(chart, kind="line")
cmp.check_vector(D, FUNNEL, "study dot across (risk ratio)", dots["x"])
cmp.check_vector(D, FUNNEL, "study dot up (standard error)", dots["y"])
cmp.check_vector(D, FUNNEL, "line at the combined effect", [x for x, y in zip(lines["x"], lines["y"]) if y != y])
cmp.check_vector(D, FUNNEL, "funnel across", [x for x, y in zip(lines["x"], lines["y"]) if y == y])
cmp.check_vector(D, FUNNEL, "funnel up", [y for y in lines["y"] if y == y])

agrees  BCG trials (metadat) | chart forest_plot (line_at=1, log_scale=) | study dot (risk ratio) [1]: simplydata 0.410938654841, R metafor escalc (RR) (metafor 5.2.1) 0.410938654841; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor escalc (RR) (metafor 5.2.1) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html
agrees  BCG trials (metadat) | chart forest_plot (line_at=1, log_scale=) | study dot (risk ratio) [2]: simplydata 0.204868154158, R metafor escalc (RR) (metafor 5.2.1) 0.204868154158; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor escalc (RR) (metafor 5.2.1) (run reference) - https://search.r-project.org/CRAN/refmans/metafor/html/escalc.html
agrees  BCG trials (metadat) | chart forest_plot (line_at=1, log_scale=) | study dot (risk ratio) [3]: simplydata 0.25974025974, R metafor escalc (RR) (metafor 5.2.1) 0.25974025974; relative difference 0.00e+00 (allowed 1e-10)
      Source: R metafor escalc (RR) (meta

In [7]:
cmp.finish()

31_combining_planning_surveys: 311 comparisons - 296 agrees, 1 explained difference – reference, 14 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,31_combining_planning_surveys,BCG trials (metadat),"combine_studies (show=""risk ratios"", test_meth...",estimate,-0.714532,-0.714500,"Viechtbauer (2010), Conducting meta-analyses i...",-3.234216e-05,5.000000e-05,absolute,3.234216e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,31_combining_planning_surveys,BCG trials (metadat),"combine_studies (show=""risk ratios"", test_meth...",standard error,0.179782,0.179800,"Viechtbauer (2010), Conducting meta-analyses i...",-1.848389e-05,5.000000e-05,absolute,1.848389e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,31_combining_planning_surveys,BCG trials (metadat),"combine_studies (show=""risk ratios"", test_meth...",low,-1.066898,-1.066900,"Viechtbauer (2010), Conducting meta-analyses i...",2.361189e-06,5.000000e-05,absolute,2.361189e-06,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,31_combining_planning_surveys,BCG trials (metadat),"combine_studies (show=""risk ratios"", test_meth...",high,-0.362167,-0.362200,"Viechtbauer (2010), Conducting meta-analyses i...",3.295449e-05,5.000000e-05,absolute,3.295449e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,31_combining_planning_surveys,BCG trials (metadat),"combine_studies (show=""risk ratios"", test_meth...",tau squared,0.313243,0.313200,"Viechtbauer (2010), Conducting meta-analyses i...",4.325814e-05,5.000000e-05,absolute,4.325814e-05,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v036.i03,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,31_combining_planning_surveys,BCG trials (metadat),chart draw_funnel_plot (log_scale=),funnel across [2],0.489421,0.489421,R metafor funnel(): combined effect +/- qnorm(...,3.037803e-09,1.000000e-06,relative,6.206933e-09,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
307,31_combining_planning_surveys,BCG trials (metadat),chart draw_funnel_plot (log_scale=),funnel across [3],2.045643,2.045643,R metafor funnel(): combined effect +/- qnorm(...,1.269717e-08,1.000000e-06,relative,6.206934e-09,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
308,31_combining_planning_surveys,BCG trials (metadat),chart draw_funnel_plot (log_scale=),funnel up [1],0.729730,0.729730,R metafor funnel(): from the largest standard ...,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
309,31_combining_planning_surveys,BCG trials (metadat),chart draw_funnel_plot (log_scale=),funnel up [2],0.000000,0.000000,R metafor funnel(): from the largest standard ...,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/meta...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
